# 1.0 - Choose Backbone (group-disjoint split)

Trains every backbone on the SAME group-disjoint split of the 10 base classes, then
compares them on test accuracy, size and inference speed, and saves only the winner.

The split is group-disjoint: no source video appears in more than one of train / val /
test. The old bake-off used the split the dataset ships with, where 78 percent of the
UCF101 groups (v_Class_gXX = one source video) had clips in several splits at once, so
a test clip's near-duplicate was in training and every accuracy was inflated. That is
why the numbers here are lower than the old 98 percent and are the honest ones.

This notebook replaces both the old bake-off and the separate teacher re-train: one
clean training path, one saved model.

## **Connect Colab**

In [1]:
import os, sys, subprocess
from getpass import getpass

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "embeddings"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [2]:
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "PASTE_YOUR_KAGGLE_KEY"

## **Download DATASET**

In [3]:
from src.bootstrap import setup
setup(drive=True, dataset=True)

Mounted at /content/drive
Drive mounted.


100%|██████████| 6.53G/6.53G [01:10<00:00, 99.1MB/s]

Extracting files...


ULAL_DATASET_ROOT = /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
ULAL_OUTPUT_ROOT  = /content/ucf101-processed/
ULAL_DATA_ROOT     = /content/UCF101
ULAL_DRIVE_PROJECT = /content/drive/MyDrive/apai
Setup complete — no config.py edit, no kernel restart.


In [4]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

ULAL_ROOT : /content/ulal
Device    : cuda
GPU       : Tesla T4 (15.6 GB)
Classes   : base=10 t1=10 t2=10 t3=10 t4=10


42

## **Preprocess data**

Confirm the shipped split leaks, build a group-disjoint one, verify it is disjoint, then
write the .pt clips. Only the 10 base classes are needed for this notebook.

In [5]:
leaking = check_group_leakage(cfg.SELECTED_CLASSES)

classes checked : 10
videos          : 1330
groups          : 250
groups in >1 split: 200

LEAKAGE: 80.0% of groups span multiple splits.
Clips from one source video are in both train and test, so every
accuracy and the probe ceiling are inflated. Re-split by group.
  v_Diving_g01                       ['train', 'val']
  v_Diving_g02                       ['test', 'train']
  v_Diving_g03                       ['test', 'train']
  v_Diving_g04                       ['test', 'train', 'val']
  v_Diving_g05                       ['train', 'val']
  v_Diving_g06                       ['test', 'train', 'val']
  v_Diving_g07                       ['test', 'train']
  v_Diving_g08                       ['test', 'train']
  v_Diving_g09                       ['train', 'val']
  v_Diving_g10                       ['train', 'val']


`build_group_split` pools every video regardless of the folder it shipped in and assigns
WHOLE groups to one split. UCF101 has 25 groups per class, so 18 / 3 / 4 gives roughly
72 / 12 / 16 percent. Deterministic by group index, no seed needed.

In [6]:
base_split = build_group_split(cfg.SELECTED_CLASSES, train_groups=18, val_groups=3)

In [7]:
clashes = describe_group_split(base_split, cfg.SELECTED_CLASSES)

split      clips   percent  clips/class
---------------------------------------
train        953     71.7%         95.3
val          165     12.4%         16.5
test         212     15.9%         21.2
---------------------------------------
TOTAL       1330                  133.0

groups: 250   groups in >1 split: 0
OK - group-disjoint.


Do not run the next cell unless the check above says `OK - group-disjoint`.

In [8]:
# BASE
preprocess_group_split(base_split, cfg.SELECTED_CLASSES, output_root=cfg.BASE_ROOT)


--- Preprocessing (group-disjoint) ---
To  : /content/UCF101/grouped_base
Classes: 10

Processing split: train



Processing split: val



Processing split: test



Done! Dataset ready at: /content/UCF101/grouped_base


In [9]:
cfg.SELECTED_CLASSES

['PlayingTabla',
 'PommelHorse',
 'JumpingJack',
 'PushUps',
 'PoleVault',
 'HorseRace',
 'HighJump',
 'Drumming',
 'HorseRiding',
 'Diving']

In [10]:
class_to_idx = {cls_name: i for i, cls_name in enumerate(cfg.SELECTED_CLASSES)}

print(f"class_to_idx:\n{class_to_idx}")

class_to_idx:
{'PlayingTabla': 0, 'PommelHorse': 1, 'JumpingJack': 2, 'PushUps': 3, 'PoleVault': 4, 'HorseRace': 5, 'HighJump': 6, 'Drumming': 7, 'HorseRiding': 8, 'Diving': 9}


## **Create Loaders**

Loaders over the group-disjoint `.pt` clips. Train is shuffled, val and test are not.

In [11]:
TRAIN_DIR = os.path.join(cfg.BASE_ROOT, "train")
VAL_DIR   = os.path.join(cfg.BASE_ROOT, "val")
TEST_DIR  = os.path.join(cfg.BASE_ROOT, "test")

train_dataset = UCF101Clips(TRAIN_DIR, class_to_idx)
val_dataset   = UCF101Clips(VAL_DIR,   class_to_idx)
test_dataset  = UCF101Clips(TEST_DIR,  class_to_idx)

In [12]:
train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=cfg.BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=cfg.BATCH_SIZE, shuffle=False)

In [13]:
print(f"{'split':<8}{'clips':>8}{'clips/class':>13}")
print("-" * 29)
print(f"{'train':<8}{len(train_dataset):>8}{len(train_dataset)/10:>13.1f}")
print(f"{'val':<8}{len(val_dataset):>8}{len(val_dataset)/10:>13.1f}")
print(f"{'test':<8}{len(test_dataset):>8}{len(test_dataset)/10:>13.1f}")
print("-" * 29)
print(f"{'TOTAL':<8}{len(train_dataset)+len(val_dataset)+len(test_dataset):>8}")

split      clips  clips/class
-----------------------------
train        953         95.3
val          165         16.5
test         212         21.2
-----------------------------
TOTAL       1330


## **Training setup**

Every backbone trains with the same epochs, learning rate and seed, so the comparison
measures the backbone and nothing else. `train_model` keeps the best-val epoch on disk,
so each model is reloaded from its best checkpoint before it is tested.

Each trial records test accuracy, best-val accuracy, parameter count, on-disk size and
inference latency. Checkpoints are written locally first; only the chosen one is copied
to Drive at the end.

In [14]:
EPOCHS         = 5
LR             = 1e-4
LOCAL_CKPT_DIR = "/content/ckpts"

os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

TRIALS = []

## **Baseline - ScratchCNN + LSTM (from scratch)**

A CNN trained from random init, no ImageNet weights. This is the reference the pretrained backbones must beat, not a real candidate.

In [15]:
scratch = ScratchCNNLSTM(num_classes=len(cfg.SELECTED_CLASSES), hidden_dim=cfg.TEACHER_HIDDEN).to(device)

In [16]:
set_seed(cfg.SEED)
hist_scratch = train_model(scratch, train_loader, val_loader,
                         num_epochs=EPOCHS, lr=LR, device=device,
                         save_path=f"{LOCAL_CKPT_DIR}/ScratchCNN.pth")

Epoch [1/5] | Train Acc: 0.3830 Loss: 1.9348 | Val Acc: 0.3697 Loss: 1.8373
  --> Model saved to /content/ckpts/ScratchCNN.pth


Epoch [2/5] | Train Acc: 0.6275 Loss: 1.3544 | Val Acc: 0.5333 Loss: 1.5362
  --> Model saved to /content/ckpts/ScratchCNN.pth


Epoch [3/5] | Train Acc: 0.7975 Loss: 0.9821 | Val Acc: 0.5394 Loss: 1.4585
  --> Model saved to /content/ckpts/ScratchCNN.pth


Epoch [4/5] | Train Acc: 0.8699 Loss: 0.7226 | Val Acc: 0.5576 Loss: 1.3471
  --> Model saved to /content/ckpts/ScratchCNN.pth


Epoch [5/5] | Train Acc: 0.9370 Loss: 0.5288 | Val Acc: 0.5273 Loss: 1.3340


In [17]:
scratch.load_state_dict(torch.load(f"{LOCAL_CKPT_DIR}/ScratchCNN.pth", map_location=device))
acc_scratch, preds_scratch, labels_scratch = test_model(scratch, test_loader, device)

Running Inference on Test Set...


Test Accuracy: 0.5896


In [18]:
lat_scratch    = measure_latency(scratch, test_loader, device)
params_scratch = sum(p.numel() for p in scratch.parameters())
size_scratch   = os.path.getsize(f"{LOCAL_CKPT_DIR}/ScratchCNN.pth") / 1e6

TRIALS.append({
    "name":        "ScratchCNN",
    "test_acc":    float(acc_scratch),
    "best_val":    float(hist_scratch["best_val_acc"]),
    "params_M":    params_scratch / 1e6,
    "size_MB":     size_scratch,
    "ms_per_clip": lat_scratch["ms_per_clip"],
})

print(TRIALS[-1])

del scratch
if torch.cuda.is_available():
    torch.cuda.empty_cache()

{'name': 'ScratchCNN', 'test_acc': 0.589622641509434, 'best_val': 0.5575757575757576, 'params_M': 103.120458, 'size_MB': 412.486134, 'ms_per_clip': 11.076627625001834}


## **ResNet18 + LSTM**

ImageNet ResNet18, frozen except layer4, 512-d features.

In [19]:
resnet18 = ResNet18LSTM(hidden_size=cfg.TEACHER_HIDDEN, num_classes=len(cfg.SELECTED_CLASSES)).to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 189MB/s]


In [20]:
set_seed(cfg.SEED)
hist_resnet18 = train_model(resnet18, train_loader, val_loader,
                         num_epochs=EPOCHS, lr=LR, device=device,
                         save_path=f"{LOCAL_CKPT_DIR}/ResNet18.pth")

Epoch [1/5] | Train Acc: 0.7555 Loss: 1.1327 | Val Acc: 0.8788 Loss: 0.4628
  --> Model saved to /content/ckpts/ResNet18.pth


Epoch [2/5] | Train Acc: 0.9643 Loss: 0.1897 | Val Acc: 0.9212 Loss: 0.2802
  --> Model saved to /content/ckpts/ResNet18.pth


Epoch [3/5] | Train Acc: 0.9864 Loss: 0.1045 | Val Acc: 0.9091 Loss: 0.2606


Epoch [4/5] | Train Acc: 0.9885 Loss: 0.0656 | Val Acc: 0.9091 Loss: 0.2728


Epoch [5/5] | Train Acc: 0.9916 Loss: 0.0597 | Val Acc: 0.9273 Loss: 0.2469
  --> Model saved to /content/ckpts/ResNet18.pth


In [21]:
resnet18.load_state_dict(torch.load(f"{LOCAL_CKPT_DIR}/ResNet18.pth", map_location=device))
acc_resnet18, preds_resnet18, labels_resnet18 = test_model(resnet18, test_loader, device)

Running Inference on Test Set...


Test Accuracy: 0.8962


In [22]:
lat_resnet18    = measure_latency(resnet18, test_loader, device)
params_resnet18 = sum(p.numel() for p in resnet18.parameters())
size_resnet18   = os.path.getsize(f"{LOCAL_CKPT_DIR}/ResNet18.pth") / 1e6

TRIALS.append({
    "name":        "ResNet18",
    "test_acc":    float(acc_resnet18),
    "best_val":    float(hist_resnet18["best_val_acc"]),
    "params_M":    params_resnet18 / 1e6,
    "size_MB":     size_resnet18,
    "ms_per_clip": lat_resnet18["ms_per_clip"],
})

print(TRIALS[-1])

del resnet18
if torch.cuda.is_available():
    torch.cuda.empty_cache()

{'name': 'ResNet18', 'test_acc': 0.8962264150943396, 'best_val': 0.9272727272727272, 'params_M': 11.967562, 'size_MB': 47.950666, 'ms_per_clip': 14.696542874986562}


## **ResNet50 + LSTM**

ImageNet ResNet50, frozen except layer4, 2048-d features. This is the backbone the rest of the pipeline (embeddings cache, freeze study, CL) is built around.

In [23]:
resnet50 = ResNet50LSTM(hidden_size=cfg.TEACHER_HIDDEN, num_classes=len(cfg.SELECTED_CLASSES), dropout_p=cfg.DROPOUT_P).to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 194MB/s]


In [24]:
set_seed(cfg.SEED)
hist_resnet50 = train_model(resnet50, train_loader, val_loader,
                         num_epochs=EPOCHS, lr=LR, device=device,
                         save_path=f"{LOCAL_CKPT_DIR}/ResNet50.pth")

Epoch [1/5] | Train Acc: 0.6401 Loss: 1.3657 | Val Acc: 0.8909 Loss: 0.5301
  --> Model saved to /content/ckpts/ResNet50.pth


Epoch [2/5] | Train Acc: 0.9381 Loss: 0.2572 | Val Acc: 0.9636 Loss: 0.2838
  --> Model saved to /content/ckpts/ResNet50.pth


Epoch [3/5] | Train Acc: 0.9759 Loss: 0.1374 | Val Acc: 0.9333 Loss: 0.2795


Epoch [4/5] | Train Acc: 0.9843 Loss: 0.0823 | Val Acc: 0.9212 Loss: 0.2711


Epoch [5/5] | Train Acc: 0.9853 Loss: 0.0674 | Val Acc: 0.9152 Loss: 0.3220


In [25]:
resnet50.load_state_dict(torch.load(f"{LOCAL_CKPT_DIR}/ResNet50.pth", map_location=device))
acc_resnet50, preds_resnet50, labels_resnet50 = test_model(resnet50, test_loader, device)

Running Inference on Test Set...


Test Accuracy: 0.9292


In [26]:
lat_resnet50    = measure_latency(resnet50, test_loader, device)
params_resnet50 = sum(p.numel() for p in resnet50.parameters())
size_resnet50   = os.path.getsize(f"{LOCAL_CKPT_DIR}/ResNet50.pth") / 1e6

TRIALS.append({
    "name":        "ResNet50",
    "test_acc":    float(acc_resnet50),
    "best_val":    float(hist_resnet50["best_val_acc"]),
    "params_M":    params_resnet50 / 1e6,
    "size_MB":     size_resnet50,
    "ms_per_clip": lat_resnet50["ms_per_clip"],
})

print(TRIALS[-1])

del resnet50
if torch.cuda.is_available():
    torch.cuda.empty_cache()

{'name': 'ResNet50', 'test_acc': 0.9292452830188679, 'best_val': 0.9636363636363636, 'params_M': 25.871946, 'size_MB': 103.807748, 'ms_per_clip': 47.924049062501695}


## **DenseNet121 + LSTM**

ImageNet DenseNet121, frozen except denseblock4 + norm5, 1024-d features.

In [27]:
densenet121 = DenseNet121LSTM(hidden_size=cfg.TEACHER_HIDDEN, num_classes=len(cfg.SELECTED_CLASSES)).to(device)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 172MB/s]


In [28]:
set_seed(cfg.SEED)
hist_densenet121 = train_model(densenet121, train_loader, val_loader,
                         num_epochs=EPOCHS, lr=LR, device=device,
                         save_path=f"{LOCAL_CKPT_DIR}/DenseNet121.pth")

Epoch [1/5] | Train Acc: 0.6842 Loss: 1.4180 | Val Acc: 0.8545 Loss: 0.7667
  --> Model saved to /content/ckpts/DenseNet121.pth


Epoch [2/5] | Train Acc: 0.9391 Loss: 0.3369 | Val Acc: 0.9152 Loss: 0.4052
  --> Model saved to /content/ckpts/DenseNet121.pth


Epoch [3/5] | Train Acc: 0.9738 Loss: 0.1434 | Val Acc: 0.8485 Loss: 0.3902


Epoch [4/5] | Train Acc: 0.9874 Loss: 0.0901 | Val Acc: 0.8424 Loss: 0.3997


Epoch [5/5] | Train Acc: 0.9948 Loss: 0.0604 | Val Acc: 0.8667 Loss: 0.3539


In [29]:
densenet121.load_state_dict(torch.load(f"{LOCAL_CKPT_DIR}/DenseNet121.pth", map_location=device))
acc_densenet121, preds_densenet121, labels_densenet121 = test_model(densenet121, test_loader, device)

Running Inference on Test Set...


Test Accuracy: 0.9434


In [30]:
lat_densenet121    = measure_latency(densenet121, test_loader, device)
params_densenet121 = sum(p.numel() for p in densenet121.parameters())
size_densenet121   = os.path.getsize(f"{LOCAL_CKPT_DIR}/DenseNet121.pth") / 1e6

TRIALS.append({
    "name":        "DenseNet121",
    "test_acc":    float(acc_densenet121),
    "best_val":    float(hist_densenet121["best_val_acc"]),
    "params_M":    params_densenet121 / 1e6,
    "size_MB":     size_densenet121,
    "ms_per_clip": lat_densenet121["ms_per_clip"],
})

print(TRIALS[-1])

del densenet121
if torch.cuda.is_available():
    torch.cuda.empty_cache()

{'name': 'DenseNet121', 'test_acc': 0.9433962264150944, 'best_val': 0.9151515151515152, 'params_M': 8.269194, 'size_MB': 33.685319, 'ms_per_clip': 46.89575874996876}


## **VGG19-BN + LSTM**

ImageNet VGG19 with batch norm, frozen except the last conv block, 512-d features.

In [31]:
vgg19 = VGG19BNLSTM(hidden_size=cfg.TEACHER_HIDDEN, num_classes=len(cfg.SELECTED_CLASSES)).to(device)

Downloading: "https://download.pytorch.org/models/vgg19_bn-c79401a0.pth" to /root/.cache/torch/hub/checkpoints/vgg19_bn-c79401a0.pth


100%|██████████| 548M/548M [00:06<00:00, 85.2MB/s]


In [32]:
set_seed(cfg.SEED)
hist_vgg19 = train_model(vgg19, train_loader, val_loader,
                         num_epochs=EPOCHS, lr=LR, device=device,
                         save_path=f"{LOCAL_CKPT_DIR}/VGG19BN.pth")

Epoch [1/5] | Train Acc: 0.6170 Loss: 1.4402 | Val Acc: 0.9091 Loss: 0.4961
  --> Model saved to /content/ckpts/VGG19BN.pth


Epoch [2/5] | Train Acc: 0.9349 Loss: 0.2782 | Val Acc: 0.8848 Loss: 0.5121


Epoch [3/5] | Train Acc: 0.9717 Loss: 0.1346 | Val Acc: 0.9333 Loss: 0.2269
  --> Model saved to /content/ckpts/VGG19BN.pth


Epoch [4/5] | Train Acc: 0.9811 Loss: 0.1068 | Val Acc: 0.9273 Loss: 0.2166


Epoch [5/5] | Train Acc: 0.9822 Loss: 0.0828 | Val Acc: 0.8970 Loss: 0.3295


In [33]:
vgg19.load_state_dict(torch.load(f"{LOCAL_CKPT_DIR}/VGG19BN.pth", map_location=device))
acc_vgg19, preds_vgg19, labels_vgg19 = test_model(vgg19, test_loader, device)

Running Inference on Test Set...


Test Accuracy: 0.9387


In [34]:
lat_vgg19    = measure_latency(vgg19, test_loader, device)
params_vgg19 = sum(p.numel() for p in vgg19.parameters())
size_vgg19   = os.path.getsize(f"{LOCAL_CKPT_DIR}/VGG19BN.pth") / 1e6

TRIALS.append({
    "name":        "VGG19BN",
    "test_acc":    float(acc_vgg19),
    "best_val":    float(hist_vgg19["best_val_acc"]),
    "params_M":    params_vgg19 / 1e6,
    "size_MB":     size_vgg19,
    "ms_per_clip": lat_vgg19["ms_per_clip"],
})

print(TRIALS[-1])

del vgg19
if torch.cuda.is_available():
    torch.cuda.empty_cache()

{'name': 'VGG19BN', 'test_acc': 0.9386792452830188, 'best_val': 0.9333333333333333, 'params_M': 20.826442, 'size_MB': 83.387417, 'ms_per_clip': 99.49676262499452}


## **Compare**

One row per backbone, sorted by test accuracy. Read accuracy first, then size and speed:
a backbone within a point or two of the best but much smaller or faster can be the better
choice.

In [35]:
results_df = pd.DataFrame(TRIALS).sort_values("test_acc", ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

       name  test_acc  best_val   params_M    size_MB  ms_per_clip
DenseNet121  0.943396  0.915152   8.269194  33.685319    46.895759
    VGG19BN  0.938679  0.933333  20.826442  83.387417    99.496763
   ResNet50  0.929245  0.963636  25.871946 103.807748    47.924049
   ResNet18  0.896226  0.927273  11.967562  47.950666    14.696543
 ScratchCNN  0.589623  0.557576 103.120458 412.486134    11.076628


In [36]:
by_acc   = results_df.iloc[0]
by_size  = results_df.sort_values("size_MB").iloc[0]
by_speed = results_df.sort_values("ms_per_clip").iloc[0]

print(f"highest accuracy : {by_acc['name']:<12} {by_acc['test_acc']:.2%}")
print(f"smallest model   : {by_size['name']:<12} {by_size['size_MB']:.0f} MB")
print(f"fastest model    : {by_speed['name']:<12} {by_speed['ms_per_clip']:.1f} ms/clip")

highest accuracy : DenseNet121  94.34%
smallest model   : DenseNet121  34 MB
fastest model    : ScratchCNN   11.1 ms/clip


## **Choose Best**

Set the winner here. Default is ResNet50: it was essentially tied for best in the old
bake-off, and it is the only backbone the downstream notebooks support without change,
because the embeddings cache, the teacher class and `BACKBONE_DIM=2048` are all built for
ResNet50's 2048-d features. Change this only if another backbone wins clearly AND you are
ready to adapt `BACKBONE_DIM` and the teacher in notebook 2.0.

In [37]:
BEST_MODEL = "ResNet50"

## **Save**

Copy only the winner to Drive. The filename `ResNet50_10C_groupsplit.pth` is what
notebook 2.0 loads, so a ResNet50 winner is drop-in.

In [38]:
src_ckpt = f"{LOCAL_CKPT_DIR}/{BEST_MODEL}.pth"
dst_dir  = f"{cfg.DRIVE_PROJECT}/resnet50/models"
dst_ckpt = f"{dst_dir}/{BEST_MODEL}_10C_groupsplit.pth"

os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src_ckpt, dst_ckpt)

print(f"best model : {BEST_MODEL}")
print(f"saved to   : {dst_ckpt}")
print(f"size       : {os.path.getsize(dst_ckpt)/1e6:.0f} MB")

best model : ResNet50
saved to   : /content/drive/MyDrive/apai/resnet50/models/ResNet50_10C_groupsplit.pth
size       : 104 MB


Verify the saved checkpoint loads into `ResNet50LSTMTeacher` exactly the way notebook 2.0
loads it, so a key mismatch surfaces here rather than in the next notebook. Only runs when
the winner is ResNet50.

In [39]:
if BEST_MODEL == "ResNet50":
    teacher = ResNet50LSTMTeacher(
        hidden_size = cfg.TEACHER_HIDDEN,
        num_classes = len(cfg.SELECTED_CLASSES),
        dropout_p   = cfg.DROPOUT_P,
    ).to(device)

    mapped = remap_teacher_checkpoint(torch.load(dst_ckpt, map_location=device))
    missing, unexpected = teacher.load_state_dict(mapped, strict=False)
    real_missing = [k for k in missing if "num_batches_tracked" not in k]

    teacher.eval()
    reload_acc, _, _ = test_model(ModelWrapper(teacher).to(device), test_loader, device)

    print(f"\nround-trip test acc : {reload_acc:.2%}")
    print(f"real missing keys   : {real_missing[:5]}")
    print("Checkpoint round-trips into the teacher." if not real_missing
          else "MISMATCH - do not continue to notebook 2.0.")
else:
    print(f"{BEST_MODEL} is not ResNet50.")
    print("Downstream (BACKBONE_DIM=2048, ResNet50LSTMTeacher, the embeddings cache) is")
    print("ResNet50-specific. Adapt those before using this checkpoint in notebook 2.0.")

  Dropped 53 num_batches_tracked buffer(s).
Running Inference on Test Set...


Test Accuracy: 0.9292

round-trip test acc : 92.92%
real missing keys   : []
Checkpoint round-trips into the teacher.


In [40]:
summary_1_0 = {
    "stage":       "1.0_choose_backbone_groupsplit",
    "classes":     cfg.SELECTED_CLASSES,
    "epochs":      EPOCHS,
    "lr":          LR,
    "best_model":  BEST_MODEL,
    "checkpoint":  dst_ckpt,
    "trials":      TRIALS,
}

os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
with open(f"{cfg.RESULTS_DIR}/stage1_choose_backbone.json", "w") as f:
    json.dump(summary_1_0, f, indent=2, default=float)

print(f"Saved -> {cfg.RESULTS_DIR}/stage1_choose_backbone.json")

Saved -> /content/drive/MyDrive/apai/results/stage1_choose_backbone.json
